In [ ]:
import json
import h3
import folium

In [29]:
nyc_outline = {
    "type": "Polygon",
    "coordinates": [[
        [-74.032502, 40.685287],
        [-74.017194, 40.729917],
        [-74.013073, 40.757015],
        [-73.949949, 40.854736],
        [-73.915666, 40.921400],
        [-73.852687, 40.901501],
        [-73.863868, 40.850075],
        [-73.875048, 40.807420],
        [-73.869846, 40.774518],
        [-73.841737, 40.760613],
        [-73.826190, 40.714454],
        [-73.834929, 40.705066],
        [-73.884179, 40.732332],
        [-73.882600, 40.701005],
        [-73.899889, 40.653419],
        [-74.040702, 40.601056],
        [-74.041878, 40.610155],
        [-74.046768, 40.637981],
        [-74.032502, 40.685287],
    ]]
}

nyc_h3 = h3.geo_to_h3shape(nyc_outline)
hexagons = h3.polygon_to_cells(nyc_h3, 11)

In [ ]:
features = []
for hex in hexagons:
    vertices = h3.cell_to_boundary(hex)
    border = [[lng, lat] for lat, lng in vertices]
    border.append(border[0])
    lat, lng = h3.cell_to_latlng(hex)

    features.append({
        "type": "Feature",
        "properties": {
            "h3_index": hex,
            "center_lat": lat,
            "center_lng": lng,
        },
        "geometry": {
            "type": "Polygon",
            "coordinates": [border],
        },
    })

nyc_h3_geojson = {
    "type": "FeatureCollection",
    "features": features,
}

with open("../Intermediate/Parcels/nyc_h3_parcels.geojson", "w") as f:
    json.dump(nyc_h3_geojson, f)

In [ ]:
center_lat = 40.765672
center_lng = -73.979179

m = folium.Map(location=[center_lat, center_lng], zoom_start=11, tiles="cartodbpositron")

folium.GeoJson(
    nyc_h3_geojson,
    style_function=lambda feature: {
        "color": "navy",
        "weight": 0.2,
        "fillOpacity": 0.1,
    },
).add_to(m)

m.save("../Intermediate/Parcels/nyc_hexagons_map.html")